In [1]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path(
    r"C:\Users\41240\OneDrive\桌面\MBS\2026 fall\Advanced Analytics\data"
)

OUTPUT = Path.cwd() / "population_outputs"
OUTPUT.mkdir(exist_ok=True)

# Read population data as text to preserve geographic identifiers.
population = pd.read_csv(
    DATA_DIR / "NJNY_Population 2021.csv.csv",
    dtype="string"
)

# Extract ZCTA identifiers and exclude the descriptive header row.
population["zcta"] = population["GEO_ID"].str.extract(
    r"US([0-9]{5})$"
)
population = population.loc[population["zcta"].notna()].copy()

population["population"] = pd.to_numeric(
    population["B01003_001E"], errors="coerce"
)
population["population_moe"] = pd.to_numeric(
    population["B01003_001M"], errors="coerce"
)

assert not population["zcta"].duplicated().any()
assert population["population"].notna().all()
assert population["population"].ge(0).all()

# Read the tab-delimited national ZCTA area file.
area = pd.read_csv(
    DATA_DIR / "2021_Gaz_zcta_national.txt",
    sep="\t",
    dtype="string"
)

area.columns = area.columns.str.strip()
area["zcta"] = area["GEOID"].str.strip()

# Convert land area from square meters to square kilometers.
area["land_area_sq_km"] = (
    pd.to_numeric(area["ALAND"], errors="coerce") / 1_000_000
)

assert not area["zcta"].duplicated().any()

# Join whole-ZCTA population and land area.
density = population[
    ["zcta", "population", "population_moe"]
].merge(
    area[["zcta", "land_area_sq_km"]],
    on="zcta",
    how="left",
    validate="one_to_one"
)

assert density["land_area_sq_km"].notna().all()
assert density["land_area_sq_km"].gt(0).all()

# Calculate people per square kilometer of land.
density["population_density_per_sq_km"] = (
    density["population"] / density["land_area_sq_km"]
)

# Record the population estimate period and area vintage.
density["population_period"] = "2017-2021 ACS 5-Year"
density["area_vintage"] = 2021

density.to_csv(
    OUTPUT / "njny_zcta_population_density_2021.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Number of ZCTAs:", len(density))
print("Saved to:", OUTPUT.resolve())
display(density.head(10))

Number of ZCTAs: 2424
Saved to: C:\Users\41240\OneDrive\文档\GitHub\practicum-project\population_outputs


,zcta,population,population_moe,land_area_sq_km,population_density_per_sq_km,population_period,area_vintage
0,07001,17132,1319,9.668679,1771.907,2017-2021 ACS 5-Year,2021
1,07002,70553,50,15.081559,4678.097271,2017-2021 ACS 5-Year,2021
2,07003,52323,77,13.776247,3798.05908,2017-2021 ACS 5-Year,2021
3,07004,7751,288,26.23198,295.47903,2017-2021 ACS 5-Year,2021
4,07005,14912,395,47.403541,314.575656,2017-2021 ACS 5-Year,2021
5,07006,26403,33,24.196964,1091.16995,2017-2021 ACS 5-Year,2021
6,07008,25085,52,11.388532,2202.65439,2017-2021 ACS 5-Year,2021
7,07009,13073,29,11.006086,1187.79737,2017-2021 ACS 5-Year,2021
8,07010,25515,32,2.477162,10300.093413,2017-2021 ACS 5-Year,2021
9,07011,40680,1682,8.061436,5046.247344,2017-2021 ACS 5-Year,2021


In [2]:
PROJECT_DIR = Path.cwd()

# Read regional commuting features.
commute = pd.read_csv(
    PROJECT_DIR / "njny_commute_outputs"
    / "njny_commute_zcta_features_2021.csv",
    dtype={"state": "string", "zcta": "string"}
)

# Read population and land-area attributes.
population = pd.read_csv(
    PROJECT_DIR / "population_outputs"
    / "njny_zcta_population_density_2021.csv",
    dtype={"zcta": "string"}
)

# Check that region keys are unique.
assert not commute.duplicated(["state", "zcta"]).any()
assert not population["zcta"].duplicated().any()

# Stop for review if a ZCTA appears in multiple study states.
assert not commute["zcta"].duplicated().any(), (
    "A ZCTA appears in multiple states; review population allocation."
)

# Keep every commuting region and attach population attributes.
regional_attributes = commute.merge(
    population,
    on="zcta",
    how="left",
    validate="many_to_one",
    indicator=True
)

# Confirm complete matching without adding rows.
assert regional_attributes["_merge"].eq("both").all()
assert len(regional_attributes) == len(commute)

regional_attributes = regional_attributes.drop(columns="_merge")

# Flag zero-population regions: per-capita rates are undefined.
regional_attributes["valid_population_denominator"] = (
    regional_attributes["population"].gt(0)
)

# Save a separate combined table; preserve both input files.
OUTPUT = PROJECT_DIR / "regional_attributes_outputs"
OUTPUT.mkdir(exist_ok=True)

regional_attributes.to_csv(
    OUTPUT / "njny_zcta_regional_attributes_2021.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Total regions:", len(regional_attributes))
print(regional_attributes.groupby("state").size())
print(
    "Regions with zero population:",
    regional_attributes["population"].eq(0).sum()
)
print("Saved to:", OUTPUT.resolve())

display(regional_attributes.head(10))

Total regions: 2424
state
NJ     598
NY    1826
dtype: int64
Regions with zero population: 59
Saved to: C:\Users\41240\OneDrive\文档\GitHub\practicum-project\regional_attributes_outputs


,state,zcta,resident_jobs_total,workplace_jobs_total,inbound_jobs_external,outbound_jobs_external,internal_jobs,inbound_jobs_cross_state,outbound_jobs_cross_state,external_inbound_share,...,cross_state_outbound_share,year,job_type,population,population_moe,land_area_sq_km,population_density_per_sq_km,population_period,area_vintage,valid_population_denominator
0,NJ,07001,7174,10997,10747,6924,250,660,824,0.977267,...,0.114859,2021,JT00,17132,1319,9.668679,1771.907000,2017-2021 ACS 5-Year,2021,True
1,NJ,07002,30537,14666,10027,25898,4639,2036,7568,0.683690,...,0.247831,2021,JT00,70553,50,15.081559,4678.097271,2017-2021 ACS 5-Year,2021,True
2,NJ,07003,25617,12899,11349,24067,1550,496,4788,0.879836,...,0.186907,2021,JT00,52323,77,13.776247,3798.059080,2017-2021 ACS 5-Year,2021,True
3,NJ,07004,4100,20386,19989,3703,397,834,626,0.980526,...,0.152683,2021,JT00,7751,288,26.231980,295.479030,2017-2021 ACS 5-Year,2021,True
4,NJ,07005,8197,4679,4157,7675,522,189,713,0.888438,...,0.086983,2021,JT00,14912,395,47.403541,314.575656,2017-2021 ACS 5-Year,2021,True
5,NJ,07006,13349,10548,9418,12219,1130,356,2278,0.892871,...,0.170649,2021,JT00,26403,33,24.196964,1091.169950,2017-2021 ACS 5-Year,2021,True
6,NJ,07008,10583,10316,9364,9631,952,846,1129,0.907716,...,0.106681,2021,JT00,25085,52,11.388532,2202.654390,2017-2021 ACS 5-Year,2021,True
7,NJ,07009,6512,4198,3849,6163,349,138,1113,0.916865,...,0.170915,2021,JT00,13073,29,11.006086,1187.797370,2017-2021 ACS 5-Year,2021,True
8,NJ,07010,10059,2665,2072,9466,593,486,3799,0.777486,...,0.377672,2021,JT00,25515,32,2.477162,10300.093413,2017-2021 ACS 5-Year,2021,True
9,NJ,07011,16285,8216,7516,15585,700,348,1707,0.914800,...,0.104820,2021,JT00,40680,1682,8.061436,5046.247344,2017-2021 ACS 5-Year,2021,True


In [3]:
#Merge population and commuting features into regional attributes